In [1]:
import numpy as np
import h5py

In [2]:
with h5py.File("/Users/berenakpinar/Desktop/lfads-analysis/output/04302025/lfads_output_bilbo_CHKDLAY_DLPFC_20250430_20ms_LFADS (2).h5", "r") as f:
    print(f)
    print(f.keys())
    #trained factors
    vf = f['valid_factors'][:].shape
    tf = f['train_factors'][:].shape
    #valid factors 

    ted = f['train_encod_data'][:].shape
    ved = f["valid_encod_data"][:].shape

    # 110 facr

    print(vf)
    print(tf)
    print(ted)
    print(ved)    

<HDF5 file "lfads_output_bilbo_CHKDLAY_DLPFC_20250430_20ms_LFADS (2).h5" (mode r)>
<KeysViewHDF5 ['train_co_means', 'train_co_stds', 'train_con_states', 'train_encod_data', 'train_factors', 'train_gen_init', 'train_gen_inputs', 'train_gen_states', 'train_ic_mean', 'train_ic_std', 'train_inds', 'train_output_params', 'train_recon_data', 'valid_co_means', 'valid_co_stds', 'valid_con_states', 'valid_encod_data', 'valid_factors', 'valid_gen_init', 'valid_gen_inputs', 'valid_gen_states', 'valid_ic_mean', 'valid_ic_std', 'valid_inds', 'valid_output_params', 'valid_recon_data']>
(88, 110, 40)
(349, 110, 40)
(349, 110, 306)
(88, 110, 306)


In [3]:
from scipy.io import loadmat

# Timing setup used by your LFADS bins
BIN_MS = 20
START_TIME_MS = 0
MIN_DELAY_MS = 450

first_ge_450_bin = int(np.ceil((MIN_DELAY_MS - START_TIME_MS) / BIN_MS))
first_ge_450_time = START_TIME_MS + first_ge_450_bin * BIN_MS

print(f"First guaranteed bin at/after {MIN_DELAY_MS} ms: bin {first_ge_450_bin} ({first_ge_450_time} ms)")
print("Meaning: bins < 450 ms are guaranteed delay; bins >= 450 ms can be delay or target/post-target depending on trial.")

# Optional: inspect behavior .mat keys to find trial target-onset field name
beh_path = "/Users/berenakpinar/Desktop/lfads-analysis/bilbo_20250430_lfads_trialparams_chk.mat"
beh = loadmat(beh_path)
keys = sorted([k for k in beh.keys() if not k.startswith("__")])
print(f"\nBehavior keys ({len(keys)}):")
print(keys)

cands = [k for k in keys if any(s in k.lower() for s in ["target", "onset", "delay", "checker", "offset"])]
print("\nTiming-related candidate keys:")
print(cands if cands else "(none found by keyword)")

First guaranteed bin at/after 450 ms: bin 23 (460 ms)
Meaning: bins < 450 ms are guaranteed delay; bins >= 450 ms can be delay or target/post-target depending on trial.

Behavior keys (7):
['trial_RTs', 'trial_action_choices', 'trial_coherences', 'trial_color_choices', 'trial_configs', 'trial_outcomes', 'trial_savetags']

Timing-related candidate keys:
(none found by keyword)


In [ ]:
# Fill this with the correct behavior key after checking candidates above.
# Example: TARGET_ONSET_KEY = "trial_target_onset_ms"
TARGET_ONSET_KEY = None

with h5py.File("/Users/berenakpinar/Desktop/lfads-analysis/output/04302025/lfads_output_bilbo_CHKDLAY_DLPFC_20250430_20ms_LFADS (2).h5", "r") as f:
    n_trials, n_time, _ = f["train_factors"].shape

times_ms = START_TIME_MS + np.arange(n_time) * BIN_MS

if TARGET_ONSET_KEY is None:
    print("Set TARGET_ONSET_KEY to your trial target-onset field name from the key list above.")
else:
    trial_target_onset_ms = np.asarray(beh[TARGET_ONSET_KEY]).reshape(-1)
    n = min(n_trials, trial_target_onset_ms.shape[0])
    trial_target_onset_ms = trial_target_onset_ms[:n]

    # phase_mask[i, t] = True if trial i is still in delay at bin t
    phase_mask_delay = times_ms[None, :] < trial_target_onset_ms[:, None]

    # Guaranteed bins from your constraint
    guaranteed_delay_bins = np.where(times_ms < MIN_DELAY_MS)[0]
    ambiguous_or_later_bins = np.where(times_ms >= MIN_DELAY_MS)[0]

    print(f"Using key: {TARGET_ONSET_KEY}")
    print(f"Trials used: {n}")
    print(f"Guaranteed-delay bins (<{MIN_DELAY_MS}ms): {guaranteed_delay_bins[0]}..{guaranteed_delay_bins[-1]} ({len(guaranteed_delay_bins)} bins)")
    print(f"First ambiguous/later bin: {ambiguous_or_later_bins[0]} ({times_ms[ambiguous_or_later_bins[0]]} ms)")

    # Per-bin fraction of trials still in delay
    frac_delay = phase_mask_delay.mean(axis=0)
    for b in [20, 22, 23, 24, 30]:
        if b < n_time:
            print(f"bin {b:>3} ({times_ms[b]:>4} ms): delay fraction = {frac_delay[b]:.3f}")